# Tuần 06: Thống kê nhập môn cho Results

Mục tiêu: dùng cleaned pre/post data để tính `n`, `mean`, `SD`, `SE`, `95% CI`, rồi viết một Results paragraph thận trọng. Tuần này ưu tiên diễn giải kết quả, không biến p-value thành mục tiêu chính.


In [1]:
from pathlib import Path
from urllib.request import urlretrieve
import hashlib

try:
    import pandas as pd
    import matplotlib
    matplotlib.use("Agg")
    matplotlib.rcParams["svg.hashsalt"] = "week06-intro-statistics"
    import matplotlib.pyplot as plt
    from scipy import stats
except ImportError as exc:
    raise SystemExit(
        "Missing package. From the project root, run: "
        "python -m pip install -r requirements.txt"
    ) from exc

THIS_WEEK = "week-06-intro-statistics-for-results"
EXPECTED_SHA256 = "175469cd9120b36a467d0e0b439f78859525841555c6a30bc5de0777edb9137a"


def find_week_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / "weeks" / THIS_WEEK,
        Path.cwd().parent,
        Path.cwd().parent / "weeks" / THIS_WEEK,
    ]
    for candidate in candidates:
        if candidate.name == THIS_WEEK and (candidate / "data/raw").exists():
            return candidate
        if (candidate / "data/raw/week06_tcsol_prepost_scores.csv").exists():
            return candidate
    raise FileNotFoundError("Cannot find the Week 06 folder. Run this notebook from the project root or the Week 06 folder.")


def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def strip_svg_whitespace(path):
    text = path.read_text(encoding="utf-8")
    clean_text = "\n".join(line.rstrip() for line in text.splitlines()) + "\n"
    path.write_text(clean_text, encoding="utf-8")

def make_ci_table(data, group_column, value_column, label_column=None, order=None, pre_column=None, post_column=None):
    rows = []
    groups = order if order is not None else sorted(data[group_column].dropna().unique())
    for group_name in groups:
        group = data[data[group_column] == group_name]
        n = len(group)
        mean_value = group[value_column].mean()
        sd_value = group[value_column].std(ddof=1)
        se_value = group[value_column].sem()
        t_critical = stats.t.ppf(0.975, n - 1) if n > 1 else float("nan")
        margin = t_critical * se_value if n > 1 else float("nan")
        row = {
            group_column: group_name,
            "n": n,
            "mean_gain": mean_value,
            "sd_gain": sd_value,
            "se_gain": se_value,
            "t_critical_95": t_critical,
            "ci95_low": mean_value - margin,
            "ci95_high": mean_value + margin,
        }
        if label_column:
            row[label_column] = group[label_column].iloc[0]
        if pre_column:
            row["pre_mean"] = group[pre_column].mean()
        if post_column:
            row["post_mean"] = group[post_column].mean()
        rows.append(row)
    columns = [group_column]
    if label_column:
        columns.append(label_column)
    columns.extend(["n"])
    if pre_column:
        columns.append("pre_mean")
    if post_column:
        columns.append("post_mean")
    columns.extend(["mean_gain", "sd_gain", "se_gain", "t_critical_95", "ci95_low", "ci95_high"])
    return pd.DataFrame(rows)[columns]

WEEK_DIR = find_week_dir()
DATA_PATH = WEEK_DIR / "data/raw/week06_tcsol_prepost_scores.csv"
TABLE_DIR = WEEK_DIR / "outputs/tables"
FIGURE_DIR = WEEK_DIR / "outputs/figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    source = "https://raw.githubusercontent.com/mtuann/tcsol-python-research/main/weeks/week-06-intro-statistics-for-results/data/raw/week06_tcsol_prepost_scores.csv"
    print("Local data not found. Downloading synthetic course dataset...")
    urlretrieve(source, DATA_PATH)

actual_sha = sha256_file(DATA_PATH)
if actual_sha != EXPECTED_SHA256:
    raise ValueError(f"Data hash mismatch. Expected {EXPECTED_SHA256}, got {actual_sha}")

print("Week folder:", WEEK_DIR)
print("Data file:", DATA_PATH)
print("SHA-256:", actual_sha)



Week folder: /Users/mitu/Desktop/work/projects/tcsol-python-research-syllabus/weeks/week-06-intro-statistics-for-results
Data file: /Users/mitu/Desktop/work/projects/tcsol-python-research-syllabus/weeks/week-06-intro-statistics-for-results/data/raw/week06_tcsol_prepost_scores.csv
SHA-256: 175469cd9120b36a467d0e0b439f78859525841555c6a30bc5de0777edb9137a


## 1. Khung nghiên cứu

Research question nhỏ: **Sau một hoạt động Hán ngữ ngắn hạn, chênh lệch trung bình là bao nhiêu và estimate này chắc đến mức nào?**

Paper connection: Week 05 tạo figure. Week 06 thêm uncertainty để câu Results không chỉ nói “cao hơn/thấp hơn”, mà nói rõ `N`, khoảng ước lượng và limitation.


In [2]:
df = pd.read_csv(DATA_PATH)
usable = df[df["usable_pre_post"] == True].copy()
usable["activity_label"] = usable["activity_focus"].map({
    "result_complements": "Result complements",
    "measure_words": "Measure words",
    "vocabulary_review": "Vocabulary review",
    "word_order": "Word order",
})

print("Raw rows:", len(df))
print("Usable rows:", len(usable))
print(usable[["learner_id", "activity_focus", "pre_score", "post_score", "gain_score"]].head(8).to_string(index=False))


Raw rows: 36
Usable rows: 25
learner_id     activity_focus  pre_score  post_score  gain_score
      S001      measure_words       62.0        75.0        13.0
      S002      measure_words       58.0        70.0        12.0
      S005 result_complements       55.0        68.0        13.0
      S006 result_complements       59.0        73.0        14.0
      S007 result_complements       61.0        74.0        13.0
      S009         word_order       64.0        73.0         9.0
      S010         word_order       63.0        72.0         9.0
      S011         word_order       65.0        74.0         9.0


## 2. Bốn con số cần đọc

- `mean`: mức gain trung bình trong sample.
- `SD`: learner trong nhóm khác nhau nhiều hay ít.
- `SE`: estimate mean ổn định đến đâu.
- `95% CI`: một khoảng ước lượng hợp lý cho mean, nếu giả định thống kê phù hợp.

Mental model: **SD nói về người học; SE/CI nói về estimate mean.**

Cảnh báo quan trọng: CI không có nghĩa là “95% learner nằm trong khoảng này”. CI nói về độ bất định quanh mean estimate.


In [3]:
order = ["result_complements", "measure_words", "vocabulary_review", "word_order"]
activity_stats = make_ci_table(
    usable,
    group_column="activity_focus",
    value_column="gain_score",
    label_column="activity_label",
    order=order,
    pre_column="pre_score",
    post_column="post_score",
).round(2)

activity_path = TABLE_DIR / "week06_activity_statistics.csv"
activity_stats.to_csv(activity_path, index=False)

print("Saved:", activity_path)
print(activity_stats.to_string(index=False))


Saved: /Users/mitu/Desktop/work/projects/tcsol-python-research-syllabus/weeks/week-06-intro-statistics-for-results/outputs/tables/week06_activity_statistics.csv
    activity_focus     activity_label  n  pre_mean  post_mean  mean_gain  sd_gain  se_gain  t_critical_95  ci95_low  ci95_high
result_complements Result complements  6     58.67      71.67      13.00     0.89     0.37           2.57     12.06      13.94
     measure_words      Measure words  5     59.60      72.20      12.60     0.55     0.24           2.78     11.92      13.28
 vocabulary_review  Vocabulary review  8     72.62      82.38       9.75     0.71     0.25           2.36      9.16      10.34
        word_order         Word order  6     64.33      73.17       8.83     0.41     0.17           2.57      8.40       9.26


## 3. Đọc bảng như người viết Results

Đừng bắt đầu bằng “significant hay không?”. Hãy đọc theo thứ tự:

1. `n`: có bao nhiêu record usable?
2. `mean_gain`: estimate chính là gì?
3. `ci95_low` đến `ci95_high`: khoảng ước lượng rộng hay hẹp?
4. limitation: sample nhỏ, synthetic data, không random assignment.

Khi so sánh activity, luôn đọc kèm group `n` và CI; bảng này chưa làm kiểm định giữa các nhóm.


In [4]:
n = len(usable)
mean_gain = usable["gain_score"].mean()
sd_gain = usable["gain_score"].std(ddof=1)
se_gain = usable["gain_score"].sem()
t_critical = stats.t.ppf(0.975, n - 1)
ci_low = mean_gain - t_critical * se_gain
ci_high = mean_gain + t_critical * se_gain

overall_summary = pd.DataFrame([{
    "raw_n": len(df),
    "usable_n": n,
    "pre_mean": usable["pre_score"].mean(),
    "post_mean": usable["post_score"].mean(),
    "mean_gain": mean_gain,
    "sd_gain": sd_gain,
    "se_gain": se_gain,
    "t_critical_95": t_critical,
    "ci95_low": ci_low,
    "ci95_high": ci_high,
    "min_gain": usable["gain_score"].min(),
    "max_gain": usable["gain_score"].max(),
}])
overall_rounded = overall_summary.round(2)
overall_path = TABLE_DIR / "week06_overall_gain_summary.csv"
overall_rounded.to_csv(overall_path, index=False)

print("Saved:", overall_path)
print(overall_rounded.to_string(index=False))


Saved: /Users/mitu/Desktop/work/projects/tcsol-python-research-syllabus/weeks/week-06-intro-statistics-for-results/outputs/tables/week06_overall_gain_summary.csv
 raw_n  usable_n  pre_mean  post_mean  mean_gain  sd_gain  se_gain  t_critical_95  ci95_low  ci95_high  min_gain  max_gain
    36        25     64.68      75.56      10.88      1.9     0.38           2.06      10.1      11.66       8.0      14.0


## 4. Figure có confidence interval

Figure Week 06 không cần màu mè hơn Week 05. Nó chỉ thêm một thứ: thanh interval để người đọc thấy estimate mean có độ bất định.


In [5]:
png_path = FIGURE_DIR / "week06_mean_gain_ci_by_activity.png"
svg_path = FIGURE_DIR / "week06_mean_gain_ci_by_activity.svg"
palette = ["#1f7a4d", "#2563eb", "#b8325f", "#b45309"]
figure_metadata = {"Date": "2026-06-03"}

fig, ax = plt.subplots(figsize=(8.6, 5.2))
y_positions = range(len(activity_stats))
means = activity_stats["mean_gain"]
error_left = means - activity_stats["ci95_low"]
error_right = activity_stats["ci95_high"] - means

ax.barh(y_positions, means, color=palette, alpha=0.88, height=0.55)
ax.errorbar(means, y_positions, xerr=[error_left, error_right], fmt="none", ecolor="#172033", elinewidth=1.8, capsize=5)
ax.set_yticks(list(y_positions))
ax.set_yticklabels([f"{label}\n(n={n})" for label, n in zip(activity_stats["activity_label"], activity_stats["n"])])


## 5. Output kiểm tra nhanh

![Figure 1: Mean gain score with 95% CI by activity focus](outputs/figures/week06_mean_gain_ci_by_activity.png)

- [Overall gain summary CSV](outputs/tables/week06_overall_gain_summary.csv)
- [Activity statistics CSV](outputs/tables/week06_activity_statistics.csv)
- [Editable SVG figure](outputs/figures/week06_mean_gain_ci_by_activity.svg)


## 6. Caption và Results paragraph frame

Figure caption frame:

> Figure 1. Mean gain score by activity focus in the synthetic Week 06 TCSOL dataset (`N = 25` usable learner records). Error bars show 95% confidence intervals around group mean gains; the figure is descriptive and does not establish causal effects.

Vietnamese thinking frame:

> Trong 25 bản ghi dùng được, chênh lệch post-test minus pre-test trung bình là `[mean]` điểm (`SD = [SD]`, `95% CI [low, high]`). Nhóm `[highest group]` có mean gain mô tả cao hơn nhóm `[lowest group]`, nhưng kết quả cần đọc thận trọng vì `[limitation]`.

English paper frame:

> In the usable Week 06 records (`N = 25`), learners showed a mean post-test minus pre-test difference of `[mean]` points (`SD = [SD]`, `95% CI [low, high]`). By activity focus, `[highest group]` had the highest descriptive mean gain, while `[lowest group]` had the lowest. These estimates should be read cautiously because `[limitation]`.


In [6]:
high = activity_stats.sort_values("mean_gain", ascending=False).iloc[0]
low = activity_stats.sort_values("mean_gain", ascending=True).iloc[0]
result_paragraph = (
    f"In the usable Week 06 records (N = {n}), learners showed a mean post-test minus pre-test "
    f"difference of {mean_gain:.2f} points (SD = {sd_gain:.2f}, 95% CI [{ci_low:.2f}, {ci_high:.2f}]). "
    f"By activity focus, {high['activity_label']} had the highest descriptive mean gain "
    f"(n = {int(high['n'])}, M = {high['mean_gain']:.2f}, 95% CI [{high['ci95_low']:.2f}, {high['ci95_high']:.2f}]), "
    f"while {low['activity_label']} had the lowest "
    f"(n = {int(low['n'])}, M = {low['mean_gain']:.2f}, 95% CI [{low['ci95_low']:.2f}, {low['ci95_high']:.2f}]). "
    "These estimates should be read cautiously because the dataset is synthetic, group sizes are small, "
    "and activity focus was not randomly assigned. The result therefore supports a descriptive claim "
    "about observed score differences, not a causal claim about teaching effectiveness."
)
print(result_paragraph)
print("\nWord count:", len(result_paragraph.split()))



In the usable Week 06 records (N = 25), learners showed a mean post-test minus pre-test difference of 10.88 points (SD = 1.90, 95% CI [10.10, 11.66]). By activity focus, Result complements had the highest descriptive mean gain (n = 6, M = 13.00, 95% CI [12.06, 13.94]), while Word order had the lowest (n = 6, M = 8.83, 95% CI [8.40, 9.26]). These estimates should be read cautiously because the dataset is synthetic, group sizes are small, and activity focus was not randomly assigned. The result therefore supports a descriptive claim about observed score differences, not a causal claim about teaching effectiveness.

Word count: 104


## 7. Optional: paired t-test

T-test trả lời câu hỏi khác với CI. CI giúp viết **estimate + uncertainty**. T-test hỏi liệu mean difference có xa 0 không dưới giả định thống kê. Với người mới, chỉ đọc output này như stretch, không dùng p-value một mình để kết luận.


In [7]:
paired_test = stats.ttest_rel(usable["post_score"], usable["pre_score"])
print("paired t statistic:", round(float(paired_test.statistic), 2))
print("p-value:", f"{float(paired_test.pvalue):.3g}")
print("df:", int(paired_test.df))
print("Stretch only: this p-value asks whether the mean paired difference is far from 0; the Results paragraph still needs mean, CI, and limitation.")


paired t statistic: 28.63
p-value: 4.58e-20
df: 24
Stretch only: this p-value asks whether the mean paired difference is far from 0; the Results paragraph still needs mean, CI, and limitation.


## 8. Transfer sang các hướng nghiên cứu

- TCSOL: report mean gain + CI cho pre/post score.
- Đối chiếu Hán-Việt: report mean difficulty rating + CI theo hiện tượng ngữ pháp.
- MT/MTPE: report mean edit time hoặc error count + CI theo system.
- Chính sách giáo dục: report mean/median coding score theo giai đoạn, nhớ nói rõ đơn vị phân tích là văn bản hay chính sách.


## 9. Exercise

1. Đọc `week06_overall_gain_summary.csv`: ghi lại usable `N`, `mean_gain`, `SD`, và `95% CI`.
2. Rerun notebook để xuất `week06_activity_statistics.csv` và CI figure.
3. Nếu muốn đổi label activity sang tiếng Việt, chỉ đổi display text bên phải trong mapping, không đổi raw key bên trái.
4. Viết một Results paragraph 120-160 từ.
5. Thêm source note và một câu limitation không nghe như lời xin lỗi.
6. Stretch: chạy paired t-test và giải thích vì sao p-value không thay thế CI.
